# 03 — Baselines: *How far do shallow features get you?*

Before we celebrate the full engine model, we fix the **reference points** every later claim is
measured against (rubric: Baseline + its error analysis). There are three, and they bracket the
problem from both sides:

| reference | what it predicts | role |
|---|---|---|
| **predict-mean** | the training-mean rating, for everyone | the trivial floor — any model must beat this |
| **copy-opponent** | the *opponent's* rating | the **leak** — strong, illegitimate, an effective *upper bound* on cheating |
| **ridge / LightGBM (no-engine)** | rating from game-shape + opening + own result | honest, shallow — how much do "surface" features carry? |

The two insights we are after:

1. **Matchmaking already "knows" your rating.** Copy-opponent scores an MAE around **80 Elo** — better
   than anything a legitimate model achieves — precisely because Lichess pairs similar ratings. That
   is not a model we can ship; it is the reason opponent-derived features are banned (notebooks 01–02).
2. **Without engine evals, surface features only get you partway.** Game shape (length, captures,
   checks), opening (`eco`), and your own result cut the error from the ~365 floor to ~293 — a real
   but modest 20% — and leave a large gap to the ~239 the engine/clock features reach in notebook 04.

In [ ]:
# --- Setup ----------------------------------------------------------------------------------
import sys, warnings
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.config import load_config
from src.plotting import set_style, save_fig
from src.model import (
    add_opponent_rating, evaluate, grouped_split, make_lgbm,
    BASELINE_FEATURES, BASELINE_CAT, BASELINE_NUM, mae, rmse,
)

warnings.filterwarnings("ignore")
cfg = load_config()
set_style()
PROC = Path(cfg["paths"]["processed"])
FIG = Path(cfg["paths"]["figures"]) / "notebook_03_baseline"   # this notebook's figure folder

def show(path):
    display(Image(filename=str(path)))

# Attach opponent_rating (for the copy-opponent reference ONLY — never a model feature).
feats = pd.read_parquet(PROC / "features_300k_v2.parquet")
games = pd.read_parquet(PROC / "games_clean_300k.parquet", columns=["game_id", "white_elo", "black_elo"])
df = add_opponent_rating(feats, games)
print("instances:", len(df), "| baseline features:", BASELINE_FEATURES)

## 1. The baseline feature set

`BASELINE_FEATURES` deliberately excludes **all** engine (centipawn / accuracy) and **all** clock
features — those are held back for the improved model so the comparison is clean. What remains is
"surface" information available without any Stockfish analysis:

- **shape:** `game_plies`, `player_moves`, `n_captures`, `n_checks`
- **opening:** `eco` (opening family, 484 distinct here)
- **own result + control:** `result`, `time_control`

If these alone predicted rating well, the whole eval-based pipeline would be unnecessary. They don't —
which is the point.

In [ ]:
# Stage 5 in one call: predict-mean, copy-opponent, ridge, and LightGBM on the SAME grouped split.
# (~1-2 minutes: fits a ridge and a LightGBM on ~486k training rows.)
results = evaluate(df, cfg)

order = ["predict_mean", "copy_opponent", "ridge", "lightgbm"]
table = pd.DataFrame(
    [(k, round(results[k]["mae"], 1), round(results[k]["rmse"], 1)) for k in order],
    columns=["model", "MAE", "RMSE"],
).set_index("model")
print(f"grouped hold-out: n_train={int(results['_meta']['n_train']):,}  n_test={int(results['_meta']['n_test']):,}")
display(table)

**Reading the table.**

- **predict-mean ≈ 365 Elo** is the floor: guess ~1640 for everyone and you are off by 365 on
  average. Any useful model lives below this line.
- **copy-opponent ≈ 81 Elo.** Astonishingly good — roughly *4.5×* better than the best legitimate
  baseline — and completely unusable. It works only because the opponent's rating is a near-copy of
  yours (notebook 01, §8). This number is the empirical case for the leakage guard.
- **ridge ≈ 298, LightGBM ≈ 293.** The honest shallow models. They beat the mean by a clear ~70–80
  Elo, and LightGBM edges ridge by exploiting non-linear interactions among the surface features — but
  both sit far above copy-opponent and, as notebook 04 shows, far above what engine features reach.

In [ ]:
# Visualise the ladder, with the full engine model (notebook 04 / reports/results.md) as a target.
FULL_MODEL_MAE = 239.1   # "+ engine/clock", 77 features, from reports/results.md (457c2a5)
names = ["predict\nmean", "ridge\n(no-engine)", "LightGBM\n(no-engine)", "copy\nopponent"]
vals = [results["predict_mean"]["mae"], results["ridge"]["mae"],
        results["lightgbm"]["mae"], results["copy_opponent"]["mae"]]
colors = ["#9aa0a6", "#4bacc6", "#2c6fbb", "#c0504d"]

fig, ax = plt.subplots(figsize=(8.5, 4.6))
bars = ax.bar(names, vals, color=colors)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.0f}", ha="center", va="bottom", fontweight="bold")
ax.axhline(FULL_MODEL_MAE, color="#3aa66f", ls="--", lw=1.5)
ax.text(3.35, FULL_MODEL_MAE + 4, f"full engine model {FULL_MODEL_MAE:.0f} (nb04)",
        color="#3aa66f", ha="right", fontsize=9)
ax.set(ylabel="MAE (Elo)", title="The baseline ladder (lower = better)")
show(save_fig(fig, "baseline_ladder", FIG))

**The copy-opponent lesson, stated plainly.** The single best predictor of your rating in this entire
dataset is *who Lichess paired you against* — not anything about how you played. A model allowed to
see the opponent would score ~81 and learn almost nothing about chess. By **banning** opponent-derived
features and splitting by `username`, we force the model to earn its accuracy from the player's own
moves. Everything from here on is measured against the honest ~293 shallow baseline, not the ~81 leak.

## 2. Error analysis of the shallow model

Metrics summarise; pictures diagnose. We refit the LightGBM baseline on the **same** grouped split
(deterministic seed, so its test set matches the table above) to inspect *how* it errs.

In [ ]:
# Refit the no-engine LightGBM on the identical grouped split to recover test-set predictions.
d = df.copy()
for col in BASELINE_CAT:                      # stabilise category codes across the split (as evaluate does)
    d[col] = d[col].astype("category")
train, test = grouped_split(d, cfg["model"]["test_size"], cfg["random_seed"])
y_tr = train["rating"].to_numpy(float)
y_te = test["rating"].to_numpy(float)

lgbm = make_lgbm(cfg).fit(train[BASELINE_FEATURES], y_tr)
pred = lgbm.predict(test[BASELINE_FEATURES])
print(f"refit LightGBM baseline: MAE {mae(y_te, pred):.1f}  (matches the table)   bias {np.mean(pred - y_te):+.1f}")

In [ ]:
# Predicted vs actual (density) — watch the vertical compression toward the mean.
fig, ax = plt.subplots(figsize=(6.8, 6))
hb = ax.hexbin(y_te, pred, gridsize=45, cmap="Blues", mincnt=1, bins="log")
lims = [600, 2800]
ax.plot(lims, lims, "k--", lw=1, label="perfect (y = x)")
ax.axhline(y_tr.mean(), color="#c0504d", ls=":", lw=1.2, label=f"train mean ~{y_tr.mean():.0f}")
ax.set(xlim=lims, ylim=lims, xlabel="Actual rating", ylabel="Predicted rating",
       title="No-engine LightGBM: predicted vs actual")
ax.legend(loc="upper left", fontsize=9)
fig.colorbar(hb, ax=ax, shrink=0.8, label="log count")
show(save_fig(fig, "baseline_pred_vs_actual", FIG))

**Reading the scatter.** The cloud is a nearly-horizontal band, far flatter than the `y = x` diagonal:
the shallow model predicts *almost everyone* into a narrow ~1450–1750 range around the training mean.
With no move-quality signal, the safest bet under an MAE loss is to hug the centre — so a true-1000
player is pulled up and a true-2400 player is pulled down. That shrinkage is not a bug in the shallow
model; it is the *rational* response to weak features, and it is the same mechanism (in milder form)
behind the tail bias of the full model in notebook 05.

In [ ]:
# Residual distribution and residual-vs-rating (the tail bias, already visible).
resid = pred - y_te                                   # + = over-predicted
edges = np.arange(600, 2700, 100)
idx = np.digitize(y_te, edges)
cen, mres = [], []
for b in range(1, len(edges)):
    m = idx == b
    if m.sum() >= 200:
        cen.append((edges[b - 1] + edges[b]) / 2); mres.append(resid[m].mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.3))
ax1.hist(resid, bins=80, color="#8064a2")
ax1.axvline(0, color="k", ls="--", lw=1)
ax1.set(xlabel="Residual (pred − actual)", ylabel="Instances",
        title=f"Residuals  (bias {resid.mean():+.0f}, MAE {np.mean(np.abs(resid)):.0f})")
ax2.axhline(0, color="k", lw=1)
ax2.plot(cen, mres, "-o", color="#c0504d", ms=4)
ax2.set(xlabel="Actual rating", ylabel="Mean residual (Elo)",
        title="Mean residual by rating: over-predict low, under-predict high")
fig.tight_layout()
show(save_fig(fig, "baseline_residuals", FIG))

**Reading the residuals.** The overall residual histogram is roughly centred (small global bias), but
the *conditional* picture on the right is the real story: the shallow model **over-predicts low-rated
players by hundreds of Elo and under-predicts high-rated players by hundreds** — a textbook
regression-to-the-mean sweep. This is the exact tail pattern notebook 05 dissects for the *full*
model, where it is much reduced but not eliminated. Seeing it here, in the baseline, establishes that
the tail bias originates in *weak single-game evidence*, not in any particular feature set.

## Takeaways

- **The floor is ~365; the leak is ~81.** Legitimate shallow models land at ~293 (LightGBM) / ~298
  (ridge) — a modest but real gain over predicting the mean.
- **Copy-opponent is the argument for the leakage guard**, not a candidate model: matchmaking encodes
  your rating far more sharply than your play does.
- **The shallow model shrinks hard toward the mean**, over-predicting the low tail and under-predicting
  the high tail — the same tail bias, in exaggerated form, that survives into the full model.
- **The ~293 → ~239 gap is the value of engine + clock features**, which notebook 04 now adds — along
  with an honest, *guaranteed* uncertainty interval.